# EDA Dataset COVID-19 — Panduan Lengkap

**Sub-CPMK:** **P4** — EDA univariat/bivariat; visualisasi dengan Matplotlib & Seaborn.

Notebook ini adalah **tutorial mandiri** (melengkapi `minggu_05.ipynb` dan `eda_titanic.ipynb`): eksplorasi data pandemi COVID-19 berbasis **time series panel** (negara × tanggal), perbandingan benua/negara, fokus **Indonesia** dan ASEAN, serta ringkasan insight untuk analitik lanjutan.

**Konteks data:** [Our World in Data — COVID-19](https://github.com/owid/covid-19-data) repositori harian kasus, kematian, dan metrik per juta penduduk. Repozitori **diarsipkan Agustus 2024** (tidak update harian); data historis tetap valid untuk belajar EDA.

**Sumber data:** CSV tidy OWID via GitHub raw. **Unduhan pertama membutuhkan internet.** Alternatif offline: salin `owid-covid-data.csv` ke `data/` dan ganti `OWID_URL`.

**Pertanyaan analitik:**
- (Q1) Bagaimana rentang waktu dan cakupan geografis data?
- (Q2) Benua/negara mana dengan kasus per juta tertinggi (snapshot)?
- (Q3) Kapan puncak gelombang kasus harian **Indonesia** dan **World**?
- (Q4) Apakah `new_cases_smoothed` (rata-rata 7 hari OWID) membantu membaca tren?
- (Q5) Hubungan `new_cases` vs `new_deaths` — kuat atau confounded?
- (Q6) Bagaimana perbandingan Indonesia vs Malaysia, Singapore, Thailand (ASEAN)?
- (Q7) Kolom mana banyak missing, dan apa implikasinya?
- (Q8) Apa bias/limitasi jika membandingkan angka antar negara?

**Referensi:**
- [DataCamp — COVID-19 EDA tutorial](https://github.com/datacamp/COVID-19-EDA-tutorial)
- [Our World in Data — covid-19-data](https://github.com/owid/covid-19-data)
- [NillsF blog — COVID analysis Jupyter](https://blog.nillsf.com/index.php/2020/07/21/how-im-doing-my-own-covid-19-data-analysis-using-jupyter-python-pandas-and-matplotlib/)
- [PseudoLab — Time Series Ch2 EDA](https://pseudo-lab.github.io/Tutorial-Book-en/chapters/en/time-series/Ch2-EDA.html)
- [Subiya101 — COVID-19 Data Analysis](https://github.com/Subiya101/COVID-19-Data-Analysis)
- Modul PDF: `modul-05.tex` (Minggu 5 praktikum)


## 0. Kerangka CRISP-DM dan kamus fitur

Fase **Data Understanding** (CRISP-DM): pahami kolom sebelum memplot. EDA **iteratif** — temuan mengarahkan cleaning (`minggu_03`), transformasi waktu, dan forecasting (`minggu_06`–`minggu_07`).

| Kolom | Arti singkat |
|-------|----------------|
| `iso_code` | Kode ISO-3 negara |
| `continent` | Benua (NA untuk agregat `World`) |
| `location` | Nama negara/wilayah atau `World` |
| `date` | Tanggal observasi (harian) |
| `total_cases` | Kasus kumulatif terkonfirmasi |
| `new_cases` | Kasus baru harian |
| `new_cases_smoothed` | Kasus harian dihaluskan (≈7 hari, OWID) |
| `total_deaths` | Kematian kumulatif |
| `new_deaths` | Kematian baru harian |
| `new_deaths_smoothed` | Kematian harian dihaluskan |
| `total_cases_per_million` | Kasus kumulatif per 1 juta penduduk |
| `new_cases_per_million` | Kasus baru harian per 1 juta |
| `total_deaths_per_million` | Kematian kumulatif per 1 juta |
| `population` | Populasi (denominator metrik per juta) |

**Variabel turunan (§2b):** `case_fatality_rate` = `total_deaths` / `total_cases` (hati-hati pembagian nol); `year_month` untuk agregasi bulanan.


## 1. Persiapan lingkungan

Import pustaka standar praktikum; konstanta URL dan kolom subset agar unduhan lab lebih ringan.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
OWID_URL = (
    "https://raw.githubusercontent.com/owid/covid-19-data/master/"
    "public/data/owid-covid-data.csv"
)
# Alternatif offline: OWID_URL = "data/owid-covid-data.csv"

COLS = [
    "iso_code", "continent", "location", "date",
    "total_cases", "new_cases", "new_cases_smoothed",
    "total_deaths", "new_deaths", "new_deaths_smoothed",
    "total_cases_per_million", "new_cases_per_million",
    "total_deaths_per_million", "population",
]

sns.set_theme(style="whitegrid", context="notebook")
%matplotlib inline

**Memuat data.** `usecols` membatasi kolom; `parse_dates` untuk operasi time series.

In [ ]:
print("Mengunduh/memuat OWID COVID-19 (internet pada unduhan pertama)...")
df_raw = pd.read_csv(OWID_URL, usecols=COLS, parse_dates=["date"])
df_raw = df_raw[df_raw["date"] >= "2020-03-01"].copy()  # fokus fase pandemi global
n_before = len(df_raw)
df_raw = df_raw.sort_values("date").drop_duplicates(subset=["location", "date"], keep="last")
print(f"Deduplicated: {n_before} -> {len(df_raw)} baris")
print("Bentuk setelah filter tanggal:", df_raw.shape)
df_raw.head()

**Cuplikan akhir, info, dan statistik deskriptif.**

In [ ]:
display(df_raw.tail())
df_raw.info()
display(df_raw.describe().round(2))

### 1b. Audit kualitas data

`sample` untuk spot-check; duplikat `(location, date)`; rentang waktu dan jumlah entitas geografis.

In [ ]:
print("Cuplikan acak 20 baris:")
display(df_raw.sample(20, random_state=RANDOM_STATE))

n_dup = df_raw.duplicated(subset=["location", "date"]).sum()
print(f"\nDuplikat (location, date): {n_dup}")
print("Rentang tanggal:", df_raw["date"].min().date(), "s/d", df_raw["date"].max().date())
print("Jumlah location unik:", df_raw["location"].nunique())
print("Baris agregat World:", (df_raw["location"] == "World").sum())

**Interpretasi (audit):** Data panel harian multi-negara; OWID menyediakan baris **`World`** teragregasi — jangan dijumlahkan ulang dengan semua negara (double counting). Duplikat `(location, date)` pada unduhan mentah diatasi dengan `drop_duplicates(keep='last')` di sel muat data.

## 2. Missing values dan wrangling ringan

Audit NA, visual missing, lalu subset negara fokus untuk EDA yang lebih cepat.

In [ ]:
missing_count = df_raw.isna().sum().sort_values(ascending=False)
missing_pct = (missing_count / len(df_raw) * 100).round(2)
missing_tbl = pd.DataFrame({"jumlah_na": missing_count, "persen": missing_pct})
missing_tbl[missing_tbl["jumlah_na"] > 0].head(10)

In [ ]:
miss = missing_tbl[missing_tbl["jumlah_na"] > 0].sort_values("jumlah_na", ascending=True)
if len(miss):
    plt.figure(figsize=(8, 4))
    sns.barplot(x=miss["jumlah_na"], y=miss.index, hue=miss.index, palette="Blues_d", legend=False)
    plt.title("Jumlah NA per kolom (top)")
    plt.xlabel("Jumlah NA")
    plt.ylabel("Kolom")
    plt.tight_layout()
    plt.show()
else:
    print("Tidak ada NA pada kolom yang dimuat.")

**Heatmap pola missing** (sample 500 baris acak agar terbaca).

In [ ]:
sample_na = df_raw.sample(min(500, len(df_raw)), random_state=RANDOM_STATE)
plt.figure(figsize=(10, 5))
sns.heatmap(sample_na.isna(), cbar=False, yticklabels=False, cmap="viridis")
plt.title("Pola NA (sample baris acak)")
plt.xlabel("Kolom")
plt.tight_layout()
plt.show()

**Interpretasi (missing):** Awal pandemi banyak NA pada `new_cases`/`new_deaths` (belum ada laporan); `continent` NA pada baris `World` dan agregat OWID lain — normal.

### 2b. Subset negara fokus

Indonesia, World, top kasus, dan beberapa negara ASEAN.

In [ ]:
# Snapshot terakhir per negara (exclude agregat OWID_*)
latest = (
    df_raw[~df_raw["iso_code"].astype(str).str.startswith("OWID", na=False)]
    .sort_values("date")
    .groupby("location", as_index=False)
    .last()
)
top_countries = latest.nlargest(8, "total_cases")["location"].tolist()
asean = ["Indonesia", "Malaysia", "Singapore", "Thailand", "World"]
focus_locations = sorted(set(top_countries + asean))

df = df_raw[df_raw["location"].isin(focus_locations)].copy()
df_idn = df[df["location"] == "Indonesia"].copy()

df["case_fatality_rate"] = np.where(
    df["total_cases"] > 0, df["total_deaths"] / df["total_cases"], np.nan
)
print("Negara/wilayah fokus:", focus_locations)
print("Bentuk df fokus:", df.shape)

## 3. EDA tabular (Part I)

Ringkasan agregat sebelum visual: benua, ranking negara, korelasi.

In [ ]:
# Snapshot terakhir subset fokus
snap = df.sort_values("date").groupby("location", as_index=False).last()
snap = snap[snap["location"] != "World"]
print("Top 10 total_cases (snapshot fokus, excl. World):")
display(
    snap.nlargest(10, "total_cases")[
        ["location", "continent", "total_cases", "total_deaths", "total_cases_per_million"]
    ].round(1)
)

In [ ]:
cont_summary = (
    snap.dropna(subset=["continent"])
    .groupby("continent", observed=True)[["total_cases_per_million", "total_deaths_per_million"]]
    .agg(["mean", "max"])
    .round(1)
)
print("Ringkasan per benua (snapshot):")
display(cont_summary)

In [ ]:
num_cols = [
    "new_cases", "new_deaths", "new_cases_smoothed", "new_deaths_smoothed",
    "total_cases_per_million", "new_cases_per_million", "case_fatality_rate",
]
corr = df[num_cols].corr()
display(corr.round(3))
print("\nKorelasi dengan new_deaths (abs terbesar):")
print(corr["new_deaths"].drop("new_deaths").sort_values(key=abs, ascending=False).head(6).round(3))

**Interpretasi (tabel, angka contoh):** `new_cases` dan `new_deaths` berkorelasi positif kuat pada data harian; metrik **per juta** mempermudah perbandingan antar negara dengan populasi berbeda. Snapshot menunjukkan negara besar (AS, India, …) dominan pada `total_cases` absolut.

## 4. EDA univariat (Part II — distribusi)

Bentuk distribusi kasus/kematian harian dan snapshot per juta.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
subset = df[df["new_cases"].notna() & (df["new_cases"] >= 0)]
sns.histplot(subset["new_cases"], bins=50, kde=True, ax=ax[0])
ax[0].set_title("Distribusi new_cases (harian, subset fokus)")
ax[0].set_xlabel("new_cases")
ax[0].set_xlim(0, subset["new_cases"].quantile(0.99))
sns.histplot(subset["new_deaths"].dropna(), bins=50, kde=True, ax=ax[1], color="coral")
ax[1].set_title("Distribusi new_deaths (harian)")
ax[1].set_xlabel("new_deaths")
ax[1].set_xlim(0, subset["new_deaths"].quantile(0.99))
plt.tight_layout()
plt.show()

**Interpretasi:** Distribusi kasus harian sangat skew — sebagian besar hari dengan angka rendah, ekor panjang saat gelombang. Memotong pada persentil 99 agar skala terbaca.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(snap["total_cases_per_million"].dropna(), kde=True, ax=ax[0])
ax[0].set_title("Snapshot total_cases_per_million")
ax[0].set_xlabel("kasus per juta")
sns.boxplot(data=snap.dropna(subset=["continent"]), x="continent", y="total_cases_per_million", ax=ax[1])
ax[1].set_title("Boxplot kasus per juta per benua")
ax[1].set_xlabel("continent")
ax[1].tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## 5. EDA bivariat — negara, benua, dan hubungan kasus–kematian

Setiap plot diikuti interpretasi (persyaratan modul Minggu 5).

In [ ]:
plot_df = df[(df["new_cases"] >= 0) & (df["new_deaths"] >= 0)].copy()
plot_df = plot_df[plot_df["location"] != "World"]
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=plot_df.sample(min(3000, len(plot_df)), random_state=RANDOM_STATE),
    x="new_cases", y="new_deaths", hue="continent", alpha=0.4, s=25,
)
plt.title("Kasus harian vs kematian harian (sample, hue: continent)")
plt.xlabel("new_cases")
plt.ylabel("new_deaths")
plt.tight_layout()
plt.show()

**Interpretasi:** Hubungan linear kasus–kematian terlihat, tetapi slope berbeda antar benua (kapasitas kesehatan, definisi kasus, usia populasi) — bukan bukti kausalitas one-to-one.

In [ ]:
compare = snap[snap["location"].isin(["Indonesia", "World", "United States", "India", "Brazil"])].copy()
compare = compare.sort_values("total_cases_per_million", ascending=False)
plt.figure(figsize=(8, 4))
sns.barplot(data=compare, x="location", y="total_cases_per_million", hue="location", legend=False, palette="viridis")
plt.title("Perbandingan total_cases_per_million (snapshot)")
plt.xlabel("location")
plt.ylabel("total_cases_per_million")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()
display(compare[["location", "total_cases_per_million", "total_deaths_per_million"]].round(1))

**Interpretasi:** **Indonesia** pada snapshot OWID terakhir berada di bawah AS/India/Brazil pada metrik per juta absolut, tetapi urutan berubah tergantung tanggal potong — selalu sebut tanggal snapshot saat melapor.

In [ ]:
asean_snap = snap[snap["location"].isin(["Indonesia", "Malaysia", "Singapore", "Thailand"])]
plt.figure(figsize=(7, 4))
sns.barplot(data=asean_snap, x="location", y="new_cases_per_million", hue="location", legend=False)
plt.title("new_cases_per_million — snapshot ASEAN (hari terakhir per negara)")
plt.xlabel("location")
plt.ylabel("new_cases_per_million")
plt.tight_layout()
plt.show()

## 6. Time series — gelombang pandemi

Visual inti COVID-19: kumulatif, harian, smoothed, dan `diff()` kumulatif→harian (PseudoLab).

In [ ]:
ts_countries = ["Indonesia", "World", "United States", "India"]
ts_df = df[df["location"].isin(ts_countries)].copy()
plt.figure(figsize=(10, 5))
for loc, sub in ts_df.groupby("location"):
    plt.plot(sub["date"], sub["total_cases"], label=loc)
plt.title("Kasus kumulatif (total_cases) — negara terpilih")
plt.xlabel("date")
plt.ylabel("total_cases")
plt.legend()
plt.tight_layout()
plt.show()

**Interpretasi:** Kurva kumulatif monoton naik; negara berpopulasi besar dan tes luas cenderung absolut tertinggi. Untuk melihat **gelombang**, gunakan kasus harian.

In [ ]:
idn = df_idn.sort_values("date")
fig, ax = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
ax[0].plot(idn["date"], idn["new_cases"], alpha=0.5, label="new_cases")
ax[0].plot(idn["date"], idn["new_cases_smoothed"], linewidth=2, label="new_cases_smoothed (7d)")
ax[0].set_title("Indonesia — kasus harian dan smoothed")
ax[0].set_ylabel("kasus")
ax[0].legend()
ax[1].plot(idn["date"], idn["new_deaths"], alpha=0.5, color="coral", label="new_deaths")
ax[1].plot(idn["date"], idn["new_deaths_smoothed"], linewidth=2, color="darkred", label="new_deaths_smoothed")
ax[1].set_title("Indonesia — kematian harian dan smoothed")
ax[1].set_xlabel("date")
ax[1].set_ylabel("kematian")
ax[1].legend()
plt.tight_layout()
plt.show()
peak_idx = idn["new_cases"].idxmax()
print("Puncak new_cases Indonesia:", idn.loc[peak_idx, "date"].date(), "->", int(idn.loc[peak_idx, "new_cases"]))

**Interpretasi:** `new_cases_smoothed` meredam noise akhir pekan/pelaporan tertunda; puncak harian Indonesia (lihat output sel di atas) identifikasi gelombang delta/Omicron secara deskriptif.

In [ ]:
# PseudoLab: diff kumulatif → harian (harus ≈ new_cases OWID)
idn_cum = df_idn.sort_values("date").set_index("date")["total_cases"]
from_diff = idn_cum.diff().fillna(idn_cum.iloc[0])
check = pd.DataFrame({"new_cases_owid": df_idn.set_index("date")["new_cases"], "from_diff": from_diff}).dropna().head()
print("Cuplikan perbandingan new_cases vs diff(total_cases):")
display(check.head(8))
plt.figure(figsize=(10, 4))
plt.plot(from_diff.index, from_diff.values, alpha=0.7, label="diff(total_cases)")
plt.plot(df_idn["date"], df_idn["new_cases"], alpha=0.7, label="new_cases OWID")
plt.title("Indonesia: rekonstruksi harian dari kumulatif (diff)")
plt.xlabel("date")
plt.ylabel("kasus")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
asean_ts = df[df["location"].isin(["Indonesia", "Malaysia", "Singapore", "Thailand"])].copy()
g = sns.FacetGrid(asean_ts, col="location", col_wrap=2, height=3, sharey=False)
g.map_dataframe(sns.lineplot, x="date", y="new_cases_smoothed")
g.set_axis_labels("date", "new_cases_smoothed")
g.set_titles(col_template="{col_name}")
g.fig.suptitle("Kasus harian dihaluskan — panel ASEAN", y=1.02)
plt.show()

**Interpretasi (ASEAN):** Timing dan amplitudo gelombang berbeda antar negara — kebijakan, testing, dan demografi berperan; hindari menyimpulkan "negara A lebih baik" hanya dari satu metrik.

In [ ]:
world = df[df["location"] == "World"].sort_values("date")
peak_w = world.loc[world["new_cases"].idxmax()]
print("Puncak new_cases World:", peak_w["date"].date(), "->", int(peak_w["new_cases"]))

## 7. EDA multivariat

Korelasi numerik dan heatmap bulanan untuk Indonesia.

In [ ]:
num = df[num_cols].dropna()
plt.figure(figsize=(7, 5))
sns.heatmap(num.corr(), annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Heatmap korelasi — fitur numerik (subset fokus)")
plt.tight_layout()
plt.show()

In [ ]:
idn_m = df_idn.copy()
idn_m["year_month"] = idn_m["date"].dt.to_period("M").astype(str)
monthly = idn_m.groupby("year_month", as_index=False)["new_cases"].mean()
pivot_ym = idn_m.assign(year=idn_m["date"].dt.year, month=idn_m["date"].dt.month)
heat = pivot_ym.pivot_table(index="month", columns="year", values="new_cases", aggfunc="mean")
plt.figure(figsize=(10, 5))
sns.heatmap(heat, cmap="YlOrRd", annot=False)
plt.title("Heatmap rata-rata new_cases — Indonesia (bulan × tahun)")
plt.xlabel("year")
plt.ylabel("month")
plt.tight_layout()
plt.show()

**Interpretasi:** Heatmap bulan×tahun menonjolkan musim gelombang (mis. kuartal tertentu dominan merah); korelasi Pearson tidak menangkap lag kematian setelah kasus — perlu analisis time series lanjutan.

In [ ]:
snap_plot = snap.dropna(subset=["total_cases_per_million", "total_deaths_per_million"])
sns.pairplot(
    snap_plot[["total_cases_per_million", "total_deaths_per_million", "case_fatality_rate", "continent"]],
    hue="continent",
    corner=True,
    plot_kws={"alpha": 0.8},
    height=2.2,
)
plt.show()

## 8. Ringkasan temuan, bias, dan hipotesis lanjutan

### Ringkasan temuan (EDA)

1. **Struktur data:** Panel harian multi-`location`; baris **`World`** adalah agregat OWID — jangan dijumlahkan dengan semua negara.
2. **Rentang waktu:** Filter dari Maret 2020; NA awal pada metrik harian umum sebelum pelaporan stabil.
3. **Skew:** `new_cases`/`new_deaths` harian sangat skew; smoothed OWID membantu membaca tren.
4. **Indonesia:** Puncak `new_cases` harian tercatat pada tanggal di output §6; profil gelombang berbeda dari World/US.
5. **Per juta:** Metrik `*_per_million` lebih adil untuk perbandingan antar negara daripada absolut.
6. **ASEAN:** Panel FacetGrid menunjukkan timing gelombang tidak serempak.
7. **Korelasi:** Kasus dan kematian harian bergerak bersama, tetapi hubungan kausal/policy tidak dapat disimpulkan dari scatter saja.
8. **Sumber:** Dataset OWID diarsipkan 2024 — cocok untuk pembelajaran, bukan dashboard operasional real-time.

### Bias dan limitasi

- **Testing & pelaporan:** Negara dengan tes sedikit melaporkan kasus lebih rendah; underreporting kematian di beberapa wilayah.
- **Definisi:** Standar "confirmed" dan "death" berubah sepanjang pandemi.
- **Populasi & kepadatan:** Metrik per juta tidak menyesuaikan struktur usia atau comorbidity.
- **Agregat World:** Campuran heterogen; detail kebijakan perlu level negara.
- **EDA iteratif (CRISP-DM):** Setelah baseline forecast, kembali ke plot residual/outlier.

### Hipotesis analitik lanjutan (tanpa implementasi penuh di notebook ini)

- Model forecast (ARIMA/Prophet) pada `new_cases_smoothed` Indonesia dengan validasi rolling window.
- Regresi `new_deaths ~ new_cases.shift(k)` untuk estimasi lag rata-rata (k harian).
- Feature kebijakan (`stringency_index`) jika kolom ditambahkan dari OWID extended CSV.

### Langkah berikutnya (alur praktikum)

1. **Cleaning & imputasi** — `minggu_03.ipynb` (NA awal, outlier harian)
2. **Encoding & scaling** — `minggu_04.ipynb`
3. **Pemodelan / evaluasi** — `minggu_06.ipynb`–`minggu_07.ipynb`

Jalankan **Kernel → Restart & Run All** untuk memastikan urutan sel konsisten.
